# ReasonLite / Notebook 3 — frozen held-out evaluation
Run all cells in order. Existing E0–E3 weights are reused; no training. The official 1,319 GSM8K test questions are evaluated for each arm. Batch size 16 (change **before** protocol lock only if memory requires; never mix batch sizes). All results are exploratory with a single seed. Do not tune on test. Use a private notebook; GPU T4 x2 or one GPU, internet on. The separate Halo-native recipe targets a supported Hopper Docker host, not Kaggle. This notebook will not install Halo or claim a Halo run.

In [ ]:
import os,sys,json,hashlib,zipfile,subprocess,shutil
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
WORK=Path('/kaggle/working'); ROOT=WORK/'ReasonLite'
assert WORK.is_dir(), 'Kaggle working directory missing'
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
os.environ['PYTHONUNBUFFERED']='1'
BATCH_SIZE=16
EXPECTED={'reasonlite/train_sft.py': 'a5547ff0be968fbfaea1c399abff996e06f1dc812ad6249e996482121add6685', 'reasonlite/train_grpo.py': '8ab790772da4b0a2c19b2418c12e2d5f407e739631d4dd5bf72921af10b4dd6a', 'reasonlite/evaluate.py': 'd20cfe3d5e572a9bb6aaedb750d070941b7510d975855e3aec2fa48badd13e41', 'reasonlite/compare.py': '2b1d765d1f8284cf2796b40bb34afad30a91be8a64851a74eb31c5b43add4bc6', 'reasonlite/__init__.py': '9ba6bd5d3f73bbc7de4c67d9856e413a2bb6be566216705d4c4cc2c79e021042', 'reasonlite/scoring.py': 'e907cb9abf179014fe7130040c06077502804d2ef46374b198b2c1c361e6abaa', 'reasonlite/data.py': '56da5e645a5673b02b16572f57858229fe1e2d1d1acc441343841d60439da9bb', 'tests/test_scoring_v2.py': '65a3a0e1f19a108915136c128e4e92e5eeb9d40fad6a8c05db04bd7ece38030d', 'tests/test_core.py': 'e234f95be2fc9542ae2d0e2acb738f5fd48b2c094ccacface50c78dd910456e8', 'configs/sft.json': 'de1496ac8ad691cc1e2dd6c0d01cb8f10c35479f95524d2e89f30455e3f36401', 'data/prepared/sft.jsonl': 'f03bbf8d1d830d0b085e47635111f4f61d2b28da54bd2667efbbd722c2966990', 'data/prepared/test.jsonl': '0c08044cb409bb6a63429e3dba31964a76a894eed39354486a51c6d46b34f559', 'data/prepared/validation.jsonl': 'f9bde6d608f710bcc967d2a46203e810dfab3310fe6d63d25a7136897b7fe155', 'data/prepared/manifest.json': '3fcde5df1054b269cc4aa4b755e499848564f62d4532e6acde9379e653aa533c', 'data/prepared/rl.jsonl': '1b3214a5eee79ba22a70985a93676cdc810139e281fd278e6dea407c597433ea', 'runs/e0_validation/predictions.jsonl': '1e9219442465dbac7adcd9c75c8cd2099e9cb48ff3ea06388fb14208913f0ea9', 'runs/e0_validation/metrics.json': '418dc6c68f9e83cb9f1eea3b718e44823312a66403552077f87bb04c3af8ac2e', 'runs/e0_validation_v2/predictions.jsonl': '559842ed60c85b7b96d8119387aa0091e8fb5b144678823afaeaac3aa008db88', 'runs/e0_validation_v2/metrics.json': 'bd3a494725b9e92f8b53a9862b29d3e56b6afe80223e27e5f632305b808a304c', 'runs/e1_validation_v2/predictions.jsonl': '35e7053f8e948734d18be908ec0aade04db451e7a90f0170b7de43e925e63174', 'runs/e1_validation_v2/metrics.json': '45ce35959db0312bbbb0a8888f8fe98881df6a2c2710b193f20f3d740c129654', 'runs/e2_validation_v1/predictions.jsonl': '39850a9c9323eb2045f976a65a76b3b0061dc01f4bf20d384fe4a2ba4042922f', 'runs/e2_validation_v1/metrics.json': '54aec3478c4dbcaeb54583a5738a0c0f5f1b3c4d29b9122a9419744c032a4e56', 'runs/e3_validation_v1/predictions.jsonl': 'b0c759f0ff4e722bfb3533d1c5a0c3ddf5934c14a46441fbca968fd005977776', 'runs/e3_validation_v1/metrics.json': '6ced32f354b2b2ff69b2a5faffe6b9142a8849690c7286d56940031a7ca36f3a', 'runs/e1_sft_v2/train_metrics.json': 'c2b7b7df2127edb100eca5d979cbfc8c49d6742e4666897c5d64bb763ac7e53b', 'runs/e1_sft_v2/training_log.json': '5d98246430be92ea1e165eb72af11dacc8c2fc7e9c1c26000cb16d23c3c2f1b8', 'runs/e1_sft_v2/adapter/tokenizer_config.json': '57fea41fa99fce34f91450643e48b3bfbb078c32c6831a3c06834f122c200510', 'runs/e1_sft_v2/adapter/merges.txt': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5', 'runs/e1_sft_v2/adapter/adapter_config.json': 'f36c1324492a136a0acdf6f5fabd3df8ccbec6a9f14285f38dc475fb7ee0f56f', 'runs/e1_sft_v2/adapter/README.md': 'c24ff37de6c5e79144231c8dbbc38c2cb277d2cb36bd062db7546abf2a48853b', 'runs/e1_sft_v2/adapter/adapter_model.safetensors': '5d885ada194f925c2f0ab34a25c503679e47c450b8d32009e775d34c2528360b', 'runs/e1_sft_v2/adapter/tokenizer.json': '9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa', 'runs/e1_sft_v2/adapter/special_tokens_map.json': 'a5dba86f1cb07d7e5b093bbea2339d2c3c4effec4d6a76d812a7271897bd151a', 'runs/e1_sft_v2/adapter/reasonlite_training.json': 'c2b7b7df2127edb100eca5d979cbfc8c49d6742e4666897c5d64bb763ac7e53b', 'runs/e1_sft_v2/adapter/added_tokens.json': '58b54bbe36fc752f79a24a271ef66a0a0830054b4dfad94bde757d851968060b', 'runs/e1_sft_v2/adapter/vocab.json': 'ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910', 'runs/grpo_smoke_v1/train_metrics.json': '27967a893f8953e8e3f074fd5f65454b1aa9b2a611afc0407679782f67bb271a', 'runs/grpo_smoke_v1/rollouts.jsonl': '0480fdf926d4210c01f32c2734cfd8da02ea83c5e2c7e83816d0ec576b88cd61', 'runs/grpo_smoke_v1/training_log.json': '58294133310bcc607e8254b7a09d38048537e25db69d6130d7dde0ae20aabf87', 'runs/grpo_smoke_v1/adapter/tokenizer_config.json': '57fea41fa99fce34f91450643e48b3bfbb078c32c6831a3c06834f122c200510', 'runs/grpo_smoke_v1/adapter/merges.txt': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5', 'runs/grpo_smoke_v1/adapter/adapter_config.json': 'f36c1324492a136a0acdf6f5fabd3df8ccbec6a9f14285f38dc475fb7ee0f56f', 'runs/grpo_smoke_v1/adapter/README.md': 'c24ff37de6c5e79144231c8dbbc38c2cb277d2cb36bd062db7546abf2a48853b', 'runs/grpo_smoke_v1/adapter/adapter_model.safetensors': '5d885ada194f925c2f0ab34a25c503679e47c450b8d32009e775d34c2528360b', 'runs/grpo_smoke_v1/adapter/tokenizer.json': '9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa', 'runs/grpo_smoke_v1/adapter/special_tokens_map.json': '76862e765266b85aa9459767e33cbaf13970f327a0e88d1c65846c2ddd3a1ecd', 'runs/grpo_smoke_v1/adapter/reasonlite_training.json': '27967a893f8953e8e3f074fd5f65454b1aa9b2a611afc0407679782f67bb271a', 'runs/grpo_smoke_v1/adapter/added_tokens.json': '58b54bbe36fc752f79a24a271ef66a0a0830054b4dfad94bde757d851968060b', 'runs/grpo_smoke_v1/adapter/vocab.json': 'ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910', 'runs/e2_grpo_correctness_v1/train_metrics.json': '6d93ad9224c5338d662548a5c74944acd40a6ee42d8fa3aef38897ef55b93e52', 'runs/e2_grpo_correctness_v1/rollouts.jsonl': '8c96cb4e31437819498467d9f7c8ae7f7c64708ca6580a01f228355b5486d7de', 'runs/e2_grpo_correctness_v1/training_log.json': '49e670cc8d111ddd51271a3bf41de7f259f57a3b327cb06716c28f7eaced4c82', 'runs/e2_grpo_correctness_v1/adapter/tokenizer_config.json': '57fea41fa99fce34f91450643e48b3bfbb078c32c6831a3c06834f122c200510', 'runs/e2_grpo_correctness_v1/adapter/merges.txt': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5', 'runs/e2_grpo_correctness_v1/adapter/adapter_config.json': 'd72443baa49d5c40d35579404ec43b8c4eed7efc79bfd1f40bf2285f461593a3', 'runs/e2_grpo_correctness_v1/adapter/README.md': 'c24ff37de6c5e79144231c8dbbc38c2cb277d2cb36bd062db7546abf2a48853b', 'runs/e2_grpo_correctness_v1/adapter/adapter_model.safetensors': '10a0722e6dbb1eacddf1fa04d1d001835c3050c5363d406545b2ecdc252ed160', 'runs/e2_grpo_correctness_v1/adapter/tokenizer.json': '9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa', 'runs/e2_grpo_correctness_v1/adapter/special_tokens_map.json': '76862e765266b85aa9459767e33cbaf13970f327a0e88d1c65846c2ddd3a1ecd', 'runs/e2_grpo_correctness_v1/adapter/reasonlite_training.json': '6d93ad9224c5338d662548a5c74944acd40a6ee42d8fa3aef38897ef55b93e52', 'runs/e2_grpo_correctness_v1/adapter/added_tokens.json': '58b54bbe36fc752f79a24a271ef66a0a0830054b4dfad94bde757d851968060b', 'runs/e2_grpo_correctness_v1/adapter/vocab.json': 'ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910', 'runs/e3_grpo_efficiency_v1/train_metrics.json': 'fe4a8cf849e571fe4b9c3c5f9bd5341c7d47f9d1c5504e602867f81e5d445448', 'runs/e3_grpo_efficiency_v1/recovery_report.json': 'dbc76d9b441eb513fe81acb558db3c22835e391efb2bd9c5d684ac9c60cd9c58', 'runs/e3_grpo_efficiency_v1/rollouts.jsonl': 'da49be6e34b5e0552f52b42a53c1b52fd84c87a7596f249bd2db66e941e702ca', 'runs/e3_grpo_efficiency_v1/original_trainer_state.txt': 'f18996c4960885b98d1aea3369cc3e70178929d6b477b7d671c68631b3577814', 'runs/e3_grpo_efficiency_v1/training_log.json': '1a30bab003f792d9cac4bfe8a9ebf13987af13c98d29f72be35f4fbab5f2612b', 'runs/e3_grpo_efficiency_v1/adapter/tokenizer_config.json': '57fea41fa99fce34f91450643e48b3bfbb078c32c6831a3c06834f122c200510', 'runs/e3_grpo_efficiency_v1/adapter/merges.txt': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5', 'runs/e3_grpo_efficiency_v1/adapter/adapter_config.json': 'b4257eb155ea53c9f3abc2a64c9bd0f00f4d3965f9a980d57a6ae687a6db32b2', 'runs/e3_grpo_efficiency_v1/adapter/README.md': 'c24ff37de6c5e79144231c8dbbc38c2cb277d2cb36bd062db7546abf2a48853b', 'runs/e3_grpo_efficiency_v1/adapter/adapter_model.safetensors': 'e1c45de45494bbb603a43ea9418739104ce5a16ab5bb1f714385caabff1e0875', 'runs/e3_grpo_efficiency_v1/adapter/tokenizer.json': '9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa', 'runs/e3_grpo_efficiency_v1/adapter/special_tokens_map.json': '76862e765266b85aa9459767e33cbaf13970f327a0e88d1c65846c2ddd3a1ecd', 'runs/e3_grpo_efficiency_v1/adapter/reasonlite_training.json': '947fffdd1396369d0d259977d5a392b56f3504499c66573ffca4e5410ea5a028', 'runs/e3_grpo_efficiency_v1/adapter/added_tokens.json': '58b54bbe36fc752f79a24a271ef66a0a0830054b4dfad94bde757d851968060b', 'runs/e3_grpo_efficiency_v1/adapter/vocab.json': 'ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910', 'runs/compare_e0_e1_nb2.json': 'd154b66181f5f467ba5f0616e204bbac6b97637778ee434bd7fd9148b5f6b023', 'runs/compare_e1_e2_nb2.json': '13ddc10a40086979ccd641d495e746040feaf02f6834da62f631549ac8330197', 'runs/compare_e1_e3_nb2.json': '623317f708a4ad0ff3054cae1c3e2986bf87b36f94ea7408e8c57cfc7baf8d80', 'runs/compare_e2_e3_nb2.json': 'db28d9f3f0594530b470d07aa30a7c898b5b8dd6180943739d9b0f855e6a05b5', 'runs/compare_e0_e2_nb2.json': '6ddbae317f4c9cc8b3429a5778df7a233de6b64351414b030fe9ce2682e84361', 'runs/compare_e0_e3_nb2.json': '0018ea1f3794ff48b4cab48c0a06887e0cf9011ce6c4e7231b9848d05037f1df'}
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
# Prefer existing session; restore only missing source or evidence from matching private input.
missing=[n for n in EXPECTED if not (ROOT/n).is_file()]
if missing:
    candidates=[WORK/'ReasonLite_Notebook2_Complete.zip']+list(Path('/kaggle/input').rglob('ReasonLite_Notebook2_Complete.zip'))
    archive=next((p for p in candidates if p.is_file()),None)
    if archive:
        with zipfile.ZipFile(archive) as z:
            for n in missing:
                payload=z.read(n); assert hashlib.sha256(payload).hexdigest()==EXPECTED[n],n
                dest=ROOT/n; dest.parent.mkdir(parents=True,exist_ok=True); dest.write_bytes(payload)
            for n in ('README.md','requirements.txt'):
                if not (ROOT/n).exists(): (ROOT/n).write_bytes(z.read(n))
    else:
        # Kaggle may unpack the uploaded ZIP automatically.
        candidates=[p.parent.parent.parent for p in Path('/kaggle/input').rglob('manifest.json') if p.parent.name=='prepared' and p.parent.parent.name=='data']
        source=next((p for p in candidates if all((p/n).is_file() for n in missing)),None)
        assert source, 'Attach the Notebook 2 complete ZIP as a private Kaggle input.'
        for n in missing:
            payload=(source/n).read_bytes(); assert hashlib.sha256(payload).hexdigest()==EXPECTED[n],n
            dest=ROOT/n; dest.parent.mkdir(parents=True,exist_ok=True); dest.write_bytes(payload)
        for n in ('README.md','requirements.txt'):
            if (source/n).is_file() and not (ROOT/n).exists(): shutil.copyfile(source/n,ROOT/n)
for n,h in EXPECTED.items(): assert sha(ROOT/n)==h,f'Original file differs: {n}'
os.chdir(ROOT)
def run(*args): subprocess.run([sys.executable,'-u',*map(str,args)],cwd=ROOT,check=True)
print('Notebook 2 source, data and evidence verified:',len(EXPECTED),'files')

## 1. Install only missing pinned dependencies, then verify GPU
Do not install a new PyTorch or CUDA. Fresh sessions may need internet to retrieve the original model.

In [ ]:
reqs=['transformers==4.51.3','peft==0.15.2','accelerate==1.6.0','datasets==3.5.0','huggingface-hub==0.30.2','trl==0.17.0']
missing=[]
for r in reqs:
    name,want=r.split('==')
    try: got=version(name)
    except PackageNotFoundError: got=None
    if got!=want: missing.append(r)
if missing: run('-m','pip','install','--upgrade-strategy','only-if-needed',*missing)
run('-c','import torch,transformers,peft; assert torch.cuda.is_available() and torch.cuda.device_count()==1; print(torch.cuda.get_device_name(0),torch.__version__,transformers.__version__,peft.__version__)')
run('-m','unittest','discover','-s','tests','-v')

## 2. Install final evaluation and report modules; freeze protocol
The test set is never used for training or reward design. E1 is the preselected efficiency candidate; E0, E2 and E3 are comparators. Batch-one validation metrics remain in the old run directories.

In [ ]:
SOURCES={'final_eval.py':'"""Frozen four-arm evaluation with atomic batch checkpoints. No training."""\nimport argparse, hashlib, json, math, os, platform, time, statistics\nfrom pathlib import Path\nfrom fractions import Fraction\nfrom .data import load_rows, messages, digest, dump_json\nfrom .scoring import grade, SCORING_VERSION\nfrom .evaluate import summarize\n\nARMS = {\'E0\':None,\'E1\':\'runs/e1_sft_v2/adapter\',\'E2\':\'runs/e2_grpo_correctness_v1/adapter\',\'E3\':\'runs/e3_grpo_efficiency_v1/adapter\'}\nVAL = dict(zip(ARMS,[\'runs/e0_validation_v2\',\'runs/e1_validation_v2\',\'runs/e2_validation_v1\',\'runs/e3_validation_v1\']))\nCAP=384\n\ndef read(p): return json.loads(Path(p).read_text(encoding=\'utf-8\'))\ndef lines(p):\n    with Path(p).open(encoding=\'utf-8\') as f: return [json.loads(l) for l in f]\ndef atomic(p,obj):\n    p=Path(p); tmp=p.with_suffix(p.suffix+\'.tmp\'); dump_json(tmp,obj); os.replace(tmp,p)\ndef completion(ids, eos):\n    for i,t in enumerate(ids):\n        if t in eos: return ids[:i+1], i+1==CAP\n    return ids, len(ids)==CAP\n\ndef verify_records(records, rows):\n    assert len(records)==len(rows)\n    for r,g in zip(records,rows):\n        assert (r[\'id\'],r[\'gold\'])==(g[\'id\'],g[\'gold\'])\n        assert all(r[k]==v for k,v in grade(r[\'output\'],Fraction(g[\'gold\']),truncated=r[\'hit_token_limit\']).items())\n        assert 0<r[\'tokens\']<=CAP\n\ndef main():\n    p=argparse.ArgumentParser(); p.add_argument(\'--arm\',choices=ARMS,required=True); p.add_argument(\'--split\',choices=[\'validation\',\'test\'],required=True); p.add_argument(\'--root\',default=\'.\'); p.add_argument(\'--out\',required=True); a=p.parse_args()\n    root=Path(a.root).resolve(); out=Path(a.out).resolve(); out.mkdir(parents=True,exist_ok=True)\n    lock=read(root/\'runs/final_protocol_v1.json\'); assert digest(root/\'reasonlite/scoring.py\')==lock[\'scorer_sha256\']; assert digest(Path(__file__))==lock[\'evaluator_sha256\']; assert SCORING_VERSION==lock[\'scoring_version\']\n    manifest, rows=load_rows(root/\'data/prepared\',a.split); assert digest(root/\'data/prepared/manifest.json\')==lock[\'manifest_sha256\']; assert manifest[\'system_prompt\']==messages(\'\')[0][\'content\']\n    B=lock[\'batch_size\']; assert lock[\'arms\']==ARMS\n    if a.split==\'validation\': rows=rows[:16]\n    else:\n        assert len(rows)==1319\n        for arm in ARMS:\n            v=read(root/\'runs/final_v1/calibration\'/arm/\'metrics.json\'); assert v[\'protocol_sha256\']==digest(root/\'runs/final_protocol_v1.json\') and v[\'n\']==16 and v[\'arm\']==arm and v[\'split\']==\'validation\'\n    import torch, transformers, peft\n    from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed\n    assert torch.cuda.is_available() and torch.cuda.device_count()==1\n    assert transformers.__version__==\'4.51.3\' and peft.__version__==\'0.15.2\'\n    set_seed(manifest[\'seed\']); tok=AutoTokenizer.from_pretrained(manifest[\'model\'],revision=manifest[\'model_revision\'],trust_remote_code=False); tok.padding_side=\'left\'\n    model=AutoModelForCausalLM.from_pretrained(manifest[\'model\'],revision=manifest[\'model_revision\'],torch_dtype=torch.float16,trust_remote_code=False).to(\'cuda\')\n    adapter=ARMS[a.arm]\n    if adapter:\n        ap=root/adapter; assert digest(ap/\'adapter_model.safetensors\')==lock[\'adapter_sha256\'][a.arm]\n        meta=read(ap/\'reasonlite_training.json\'); assert meta[\'model_revision\']==manifest[\'model_revision\'] and meta[\'data_manifest_sha256\']==lock[\'manifest_sha256\']\n        model=peft.PeftModel.from_pretrained(model,str(ap))\n    model.eval(); eos=model.generation_config.eos_token_id; eos=[eos] if isinstance(eos,int) else list(eos or [])\n    assert eos and tok.pad_token_id is not None\n    protocol={\'arm\':a.arm,\'split\':a.split,\'n\':len(rows),\'split_sha256\':manifest[\'files\'][a.split][\'sha256\'],\'protocol_sha256\':digest(root/\'runs/final_protocol_v1.json\'),\'model_revision\':manifest[\'model_revision\'],\'adapter_sha256\':lock[\'adapter_sha256\'][a.arm],\'scoring_version\':SCORING_VERSION,\'batch_size\':B,\'max_new_tokens\':CAP,\'decoding\':\'greedy\',\'dtype\':\'torch.float16\',\'device\':\'cuda\',\'gpu\':torch.cuda.get_device_name(0),\'torch\':torch.__version__,\'transformers\':transformers.__version__,\'peft\':peft.__version__,\'python\':platform.python_version(),\'eos_token_ids\':eos,\'pad_token_id\':tok.pad_token_id,\'attention_implementation\':model.config._attn_implementation,\'timing\':\'synchronized batch wall time; per-example seconds are amortized, not request latency\'}\n    if a.split==\'test\':\n        for arm in ARMS:\n            v=read(root/\'runs/final_v1/calibration\'/arm/\'metrics.json\')\n            for k in (\'gpu\',\'torch\',\'transformers\',\'peft\',\'python\',\'dtype\',\'batch_size\',\'attention_implementation\',\'eos_token_ids\',\'pad_token_id\',\'max_new_tokens\'):\n                assert v[k]==protocol[k], f\'Calibration environment differs: {arm}/{k}\'\n    pp=out/\'protocol.json\'\n    if pp.exists(): assert read(pp)==protocol, \'Resume environment or protocol differs; use a fresh run directory for all arms\'\n    else: atomic(pp,protocol)\n    @torch.inference_mode()\n    def generate(questions,cap):\n        prompts=[tok.apply_chat_template(messages(q),tokenize=False,add_generation_prompt=True) for q in questions]\n        inputs=tok(prompts,return_tensors=\'pt\',padding=True,add_special_tokens=False).to(\'cuda\')\n        torch.cuda.synchronize(); start=time.perf_counter()\n        result=model.generate(**inputs,max_new_tokens=cap,do_sample=False,pad_token_id=tok.pad_token_id,temperature=None,top_p=None,top_k=None)\n        torch.cuda.synchronize(); elapsed=time.perf_counter()-start\n        return result[:,inputs[\'input_ids\'].shape[1]:].tolist(),elapsed\n    generate([\'What is 2 plus 2?\']*B,16); torch.cuda.reset_peak_memory_stats()\n    allrows=[]; batchtimes=[]; maxmem=[]\n    for start in range(0,len(rows),B):\n        subset=rows[start:start+B]; bp=out/f\'batch_{start:05d}.json\'\n        if bp.exists():\n            batch=read(bp); assert batch[\'protocol_sha256\']==digest(pp); verify_records(batch[\'rows\'],subset)\n        else:\n            ids,elapsed=generate([r[\'question\'] for r in subset],CAP); records=[]\n            for g,seq in zip(subset,ids):\n                seq,hit=completion(seq,eos); text=tok.decode(seq,skip_special_tokens=True)\n                records.append({\'id\':g[\'id\'],\'gold\':g[\'gold\'],\'output\':text,\'tokens\':len(seq),\'seconds\':elapsed/len(subset),\'hit_token_limit\':hit,**grade(text,Fraction(g[\'gold\']),truncated=hit)})\n            verify_records(records,subset)\n            batch={\'protocol_sha256\':digest(pp),\'rows\':records,\'seconds\':elapsed,\'peak_allocated_gpu_mib\':torch.cuda.max_memory_allocated()/2**20}; atomic(bp,batch)\n        allrows.extend(batch[\'rows\']); batchtimes.append(batch[\'seconds\']); maxmem.append(batch[\'peak_allocated_gpu_mib\']); print(f\'{a.arm} {a.split}: {len(allrows)}/{len(rows)}\',flush=True)\n    verify_records(allrows,rows); m=summarize(allrows); m[\'amortized_seconds_per_question\']=m.pop(\'mean_latency_s\'); m[\'p95_amortized_seconds_per_question\']=m.pop(\'p95_latency_s\'); m.update(protocol); m.update({\'generation_wall_seconds\':sum(batchtimes),\'questions_per_second\':len(rows)/sum(batchtimes),\'peak_allocated_gpu_mib\':max(maxmem)})\n    tmp=out/\'predictions.jsonl.tmp\'\n    with tmp.open(\'w\',encoding=\'utf-8\') as f:\n        for r in allrows: f.write(json.dumps(r,ensure_ascii=False,allow_nan=False)+\'\\n\')\n    os.replace(tmp,out/\'predictions.jsonl\'); m[\'predictions_sha256\']=digest(out/\'predictions.jsonl\'); atomic(out/\'metrics.json\',m); print(json.dumps(m,indent=2))\n\nif __name__==\'__main__\': main()\n','final_report.py':'"""Paired evaluation and audit reporting without model selection on test."""\nimport math,json,csv,itertools,statistics\nfrom pathlib import Path\nfrom collections import defaultdict\nfrom .final_eval import ARMS,VAL,read,lines,verify_records\nfrom .data import load_rows,digest,dump_json\n\ndef exact_mcnemar(b,c):\n    n=b+c\n    return min(1.0,2*sum(math.comb(n,k) for k in range(min(b,c)+1))/2**n) if n else 1.0\n\ndef holm(p):\n    order=sorted(range(len(p)),key=lambda i:p[i]); out=[0.]*len(p); running=0.\n    for rank,i in enumerate(order):\n        running=max(running,min(1.,p[i]*(len(p)-rank))); out[i]=running\n    return out\n\ndef paired(x,y):\n    assert [(r[\'id\'],r[\'gold\']) for r in x]==[(r[\'id\'],r[\'gold\']) for r in y]\n    b=sum(not u[\'correct\'] and v[\'correct\'] for u,v in zip(x,y)); c=sum(u[\'correct\'] and not v[\'correct\'] for u,v in zip(x,y)); n=len(x)\n    return {\'n\':n,\'wrong_to_correct\':b,\'correct_to_wrong\':c,\'accuracy_delta_pp\':100*(b-c)/n,\'mcnemar_p\':exact_mcnemar(b,c),\'mean_token_delta\':statistics.mean(v[\'tokens\']-u[\'tokens\'] for u,v in zip(x,y))}\n\ndef audit(root):\n    result={}\n    for arm,path in [(\'E2\',\'e2_grpo_correctness_v1\'),(\'E3\',\'e3_grpo_efficiency_v1\')]:\n        r=root/\'runs\'/path; logs=read(r/\'training_log.json\'); rolls=lines(r/\'rollouts.jsonl\'); groups=defaultdict(list)\n        for row in rolls: groups[row[\'id\']].append(row)\n        assert len(groups)==64 and all(len(g)==4 for g in groups.values())\n        result[arm]={\'rollouts\':len(rolls),\'all_wrong_groups\':sum(not any(x[\'correct\'] for x in g) for g in groups.values()),\'all_correct_groups\':sum(all(x[\'correct\'] for x in g) for g in groups.values()),\'variable_reward_groups\':sum(len(set(x[\'reward\'] for x in g))>1 for g in groups.values()),\'nonfinite_gradient_steps\':[x.get(\'step\') for x in logs if \'grad_norm\' in x and (x[\'grad_norm\'] is None or not math.isfinite(x[\'grad_norm\']))],\'training_seconds\':read(r/\'train_metrics.json\')[\'train_seconds\']}\n    return result\n\ndef main():\n    root=Path.cwd(); base=root/\'runs/final_v1\'; report=base/\'report\'; report.mkdir(exist_ok=True); lock=read(root/\'runs/final_protocol_v1.json\'); _,gold=load_rows(root/\'data/prepared\',\'test\'); metrics={}; predictions={}; calibration={}\n    for arm in ARMS:\n        path=base/\'test\'/arm; m=read(path/\'metrics.json\'); assert m[\'protocol_sha256\']==digest(root/\'runs/final_protocol_v1.json\') and m[\'n\']==1319 and m[\'split\']==\'test\' and m[\'arm\']==arm and m[\'adapter_sha256\']==lock[\'adapter_sha256\'][arm]\n        assert digest(path/\'predictions.jsonl\')==m[\'predictions_sha256\']; r=lines(path/\'predictions.jsonl\'); verify_records(r,gold); metrics[arm]=m; predictions[arm]=r\n        c=lines(base/\'calibration\'/arm/\'predictions.jsonl\'); old=lines(root/VAL[arm]/\'predictions.jsonl\')[:16]; calibration[arm]={\'n\':16,\'exact_text_agreement\':sum(x[\'output\']==y[\'output\'] for x,y in zip(c,old))/16,\'correctness_agreement\':sum(x[\'correct\']==y[\'correct\'] for x,y in zip(c,old))/16,\'batched_accuracy\':sum(x[\'correct\'] for x in c)/16,\'original_accuracy\':sum(x[\'correct\'] for x in old)/16}\n    keys=[\'protocol_sha256\',\'gpu\',\'torch\',\'transformers\',\'peft\',\'python\',\'dtype\',\'batch_size\',\'attention_implementation\',\'eos_token_ids\',\'pad_token_id\',\'max_new_tokens\',\'split_sha256\']\n    for k in keys: assert all(metrics[x][k]==metrics[\'E0\'][k] for x in ARMS),k\n    comparisons=[]\n    for x,y in itertools.combinations(ARMS,2): comparisons.append({\'before\':x,\'after\':y,**paired(predictions[x],predictions[y])})\n    for c,p in zip(comparisons,holm([c[\'mcnemar_p\'] for c in comparisons])): c[\'holm_p\']=p\n    diagnostics=audit(root); dump_json(report/\'summary.json\',{\'metrics\':metrics,\'comparisons\':comparisons,\'calibration\':calibration,\'grpo_diagnostics\':diagnostics,\'primary_candidate_selected_on_validation\':\'E1\',\'halo_native_status\':\'separate experiment; no execution claimed\'})\n    fields=[\'arm\',\'n\',\'correct\',\'accuracy\',\'strict_accuracy\',\'format_rate\',\'invalid_rate\',\'mean_tokens\',\'correct_per_1000_generated_tokens\',\'token_limit_rate\',\'questions_per_second\',\'amortized_seconds_per_question\',\'peak_allocated_gpu_mib\']\n    with (report/\'metrics.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fields,extrasaction=\'ignore\'); w.writeheader(); w.writerows(metrics.values())\n    with (report/\'paired.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,list(comparisons[0])); w.writeheader(); w.writerows(comparisons)\n    text=[\'# ReasonLite: frozen held-out evaluation\',\'\', \'All four pre-existing checkpoints evaluated on the same 1,319 official GSM8K test questions. E1 was selected as the efficiency candidate using development validation only. No further model selection or tuning was based on test results.\',\'\', \'| Arm | Accuracy (Wilson 95% CI) | Mean tokens | Correct / 1k tokens | Format rate |\',\'|---|---:|---:|---:|---:|\']\n    for arm,m in metrics.items(): text.append(f"| {arm} | {m[\'accuracy\']:.2%} ({m[\'wilson_95\'][0]:.2%}–{m[\'wilson_95\'][1]:.2%}) | {m[\'mean_tokens\']:.2f} | {m[\'correct_per_1000_generated_tokens\']:.3f} | {m[\'format_rate\']:.2%} |")\n    text+=[\'\',\'## Paired tests\',\'\', \'| Contrast | Delta (pp) | Exact McNemar p | Holm-adjusted p |\',\'|---|---:|---:|---:|\']\n    for c in comparisons: text.append(f"| {c[\'before\']} → {c[\'after\']} | {c[\'accuracy_delta_pp\']:+.2f} | {c[\'mcnemar_p\']:.4g} | {c[\'holm_p\']:.4g} |")\n    text+=[\'\',\'## Limitations\',\'\', \'- Single training seed; confidence intervals and paired tests quantify question-level uncertainty, not training-seed variability.\',\'- Scorer v2 was revised on development outputs; numeric answer correctness is not proof of correct reasoning. GSM8K may overlap pretraining.\',\'- Batched greedy inference can differ from batch-one generation. Calibration results are recorded separately; batched throughput is not single-request latency.\',\'- GRPO arms are small exploratory pilots; KL disabled; E3 had a nonfinite gradient with finite saved adapter weights. Length shaping used retokenized lengths and a format bonus.\',\'- Halo native training is separate and not claimed completed.\',\'\']\n    (report/\'REPORT.md\').write_text(\'\\n\'.join(text),encoding=\'utf-8\'); print(\'\\n\'.join(text))\n    try:\n        import matplotlib.pyplot as plt\n        for metric,label in [(\'accuracy\',\'Answer accuracy\'),(\'mean_tokens\',\'Mean generated tokens\'),(\'correct_per_1000_generated_tokens\',\'Correct answers / 1k tokens\')]:\n            fig,ax=plt.subplots(figsize=(7,4)); ax.bar(list(metrics),[m[metric] for m in metrics.values()]); ax.set_ylabel(label); ax.set_title(\'GSM8K held-out test\'); fig.tight_layout(); fig.savefig(report/(metric+\'.png\'),dpi=160); plt.close(fig)\n    except ImportError: print(\'Matplotlib unavailable; CSV and report are complete.\')\n\nif __name__==\'__main__\': main()\n'}
for n,s in SOURCES.items():
    p=ROOT/'reasonlite'/n
    if p.exists(): assert p.read_bytes()==s.encode(),n
    else: p.write_bytes(s.encode())
lock={'arms': {'E0': None, 'E1': 'runs/e1_sft_v2/adapter', 'E2': 'runs/e2_grpo_correctness_v1/adapter', 'E3': 'runs/e3_grpo_efficiency_v1/adapter'}, 'adapter_sha256': {'E0': None, 'E1': '5d885ada194f925c2f0ab34a25c503679e47c450b8d32009e775d34c2528360b', 'E2': '10a0722e6dbb1eacddf1fa04d1d001835c3050c5363d406545b2ecdc252ed160', 'E3': 'e1c45de45494bbb603a43ea9418739104ce5a16ab5bb1f714385caabff1e0875'}, 'manifest_sha256': '3fcde5df1054b269cc4aa4b755e499848564f62d4532e6acde9379e653aa533c', 'scorer_sha256': 'e907cb9abf179014fe7130040c06077502804d2ef46374b198b2c1c361e6abaa', 'evaluator_sha256': '3bb4c075e4004a889e74d618e1605013f5c17116571435a7711f11cd2c011da4', 'scoring_version': 'final-answer-v2', 'selection': 'E1 primary efficiency candidate based on validation; report all E0–E3 regardless of test results', 'test_n': 1319, 'batch_size': 16, 'max_new_tokens': 384, 'decoding': 'greedy', 'seed': 42, 'calibration': 'first 16 validation items per arm; record differences vs previous single-example evaluation; do not tune on test', 'training': 'none; all existing checkpoints frozen', 'halo': 'separate supported host recipe; not part of E0–E3 test'}
lock['batch_size']=BATCH_SIZE
assert isinstance(BATCH_SIZE,int) and 1<=BATCH_SIZE<=32
p=ROOT/'runs/final_protocol_v1.json'
if p.exists(): assert json.loads(p.read_text())==lock,'Protocol already frozen. Do not change settings after evaluation starts.'
else: p.write_text(json.dumps(lock,indent=2)+'\n')
run('-m','compileall','-q','reasonlite')
run('-c',"from reasonlite.final_eval import completion; from reasonlite.final_report import exact_mcnemar,holm; assert completion([11,9,0],[9])==([11,9],False); assert exact_mcnemar(0,5)==0.0625; assert holm([.01,.04,.03])==[.03,.06,.06]; print('Final evaluation checks passed')")
print('Frozen protocol:',sha(p))
run('-c',"from pathlib import Path; from reasonlite.final_report import audit; import json; print(json.dumps(audit(Path.cwd()),indent=2))")

## 3. Calibration — first 16 development questions per arm
A small check of batched generation versus the prior batch-one results. Differences are possible with fp16 and different padding; retain them rather than editing the scorer. This is not final test evaluation.

In [ ]:
from reasonlite.final_eval import ARMS,VAL,lines,read
base=ROOT/'runs/final_v1'
def evaluate(arm,split):
    folder='calibration' if split=='validation' else 'test'
    run('-m','reasonlite.final_eval','--arm',arm,'--split',split,'--out',base/folder/arm)
for arm in ARMS:
    evaluate(arm,'validation')
    new=lines(base/'calibration'/arm/'predictions.jsonl'); old=lines(ROOT/VAL[arm]/'predictions.jsonl')[:16]
    assert [(x['id'],x['gold']) for x in new]==[(x['id'],x['gold']) for x in old]
    print(arm,'text agreement',sum(x['output']==y['output'] for x,y in zip(new,old)),'/16; correctness agreement',sum(x['correct']==y['correct'] for x,y in zip(new,old)),'/16')
print('All calibration runs completed.')
for arm in ARMS:
    m=read(base/'calibration'/arm/'metrics.json')
    print(arm, 'rough test minutes at calibration throughput:', round(1319/m['questions_per_second']/60,1))
print('Calibration timing is a rough estimate only; long batches may take longer.')

## 4. Final evaluation — four independent cells
Run all four cells. Each batch is written atomically and verified on resume. If interrupted, rerun the same cell after restoring the saved working directory. Never switch batch sizes or hardware mid-run; if hardware differs, start a new full comparison. No test-based tuning. A completed arm is reconstructed from saved batches without generating them again (model load/warmup still occurs).

In [ ]:
evaluate('E0','test')

In [ ]:
evaluate('E1','test')

In [ ]:
evaluate('E2','test')

In [ ]:
evaluate('E3','test')

## 5. Report and evidence export
Exact paired McNemar tests with Holm correction, Wilson confidence intervals, throughput metrics, GRPO diagnostics and figures. The report preserves limitations and negative results. Download ZIP before ending session.

In [ ]:
run('-m','reasonlite.final_report')
from IPython.display import display, FileLink
out=WORK/'ReasonLite_Notebook3_Complete.zip'
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
    # Prior complete evidence plus new evaluation (omit duplicated atomic batch files; final predictions retained).
    names=set(EXPECTED)|{'README.md','requirements.txt','reasonlite/final_eval.py','reasonlite/final_report.py','runs/final_protocol_v1.json'}
    names.update(str(p.relative_to(ROOT)) for p in base.rglob('*') if p.is_file() and not p.name.startswith('batch_') and not p.name.endswith('.tmp'))
    for n in sorted(names):
        p=ROOT/n
        if p.is_file(): z.write(p,n)
with zipfile.ZipFile(out) as z: assert z.testzip() is None
print('NOTEBOOK 3 EVALUATION COMPLETE:',out,round(out.stat().st_size/2**20,2),'MiB')
display(FileLink(str(out)))

## 6. Separate Halo-native milestone
Download `ReasonLite_Halo_Native_Smoke.zip` from the chat. Run on a compatible Hopper host using the included `run_halo.sh` (requires Docker and NVIDIA Container Toolkit). The recipe is a two-step native Halo LoRA smoke, not part of E0–E3. Do not claim completion until logs and trained weights are verified. Official installation: https://whitecircle.com/halo/docs/installation . No Halo installation is attempted here.